Get the PCLR features.

In [2]:
# imports
import os
import numpy as np
import h5py
from utils import utils as u
from utils.build_model import *
import pandas as pd
from tensorflow.keras.models import load_model, Model
import time
import math
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

The trained models available.

In [5]:
# some trained models
class CAPE:
    def __init__(self, dataset='ptb', name = 'mimic_test', 
                size=0, model = 'CAPE_mimic', model_params=None,
                leads_in=None, leads_out=None):
        """
        Supervised trainig configuraion for downstream tasks. The code assumes the directory structure described in readme.txt and the configuration for the CAPE model.
        Modify accordingly for your model and data.
        """
        self.model = model # trained model name
        self.dataset = dataset #name of dataset to generate features for
        self.model_path = os.path.join('models', model) # model located in models dir
        self.size = size # if generating features for a subset
        if self.model=='CAPE_mimic_2': # dict for parameters related to model
            model_params={} 
            model_params['num_leads']=8 # lead/channels for model input
            model_params['num_samples']=2800 # required num samples 
            model_params['num_feats']=256 # number of features generated
            model_params['power']=0 # if a preprocessing taking power of input
            model_params['sampling_rate']=400 # required sampling rate
            model_params['epoch']=0 # 0 if loading best model
        self.model_params = model_params
        data_params ={} # dict for parameters related to ecg data 
        data_params['filt']= 2 if dataset == 'ptb' else 0 # 0: no filtering; 1: only bandpass filtering [0.5, 100]; 2: bandpass and notch filtering (60)
        data_params['pad']= 0 # 0:if no padding; 1:if to pad with zeros if samples < num_samples;1
        data_params['size']=0 # 0:if using all ecgs; otherwise: provide size
        data_params['data_key']='ECGs' # h5 dataset for ecgs 
        self.data_params = data_params
        feat_dir = 'feats' # dir for feats
        fname = f'CAPE_feats_{self.dataset}_{self.model}' # name of feats file
        if self.size: fname = fname + f'_{self.data_size}datasize'
        self.feat_path = os.path.join(feat_dir, self.dataset, f'{fname}.h5')
        self.data_info = u.get_data_info() # dict with basic info of datasets included in current research. Update dict if using other datasets.
        self.data_info['path']['ptb'] = 'ptb' # dataset dir name
        self.data_dir = os.path.join('data', 'ECGdata', self.data_info['path'][dataset])
        self.ecg_model=get_model_embed(self.model_path, num_channels=self.model_params['num_leads'], # load the backbone from the saved model
                            num_samples=self.model_params['num_samples'], 
                            num_feats= self.model_params['num_feats'], epoch=self.model_params['epoch'])
        self.data = h5py.File(os.path.join(self.data_dir, 'ecg_data.h5'), 'r')[data_params['data_key']] # pointer to h5 dataset
        print(f'ecg data of shape = {self.data.shape} found at {self.data_dir}')
        if self.size ==0: self.size = len(self.data)
        elif self.size < len(self.data): self.size = len(ecgs)
        self.sl = 1000 # size of slices used for feature generation to reduce memory requirement.
        self.num_slices = math.ceil(self.size/self.sl) # total slices
        print(f'The data of size {self.size} will be processed in {self.num_slices} slices of {self.sl}')
        if not leads_in:
            if self.data.shape[-1] == 8:
              leads_in = u.ECG_8_LEADS
            elif self.data.shape[-1] == 12:
              leads_in = u.ECG_12_LEADS
        if not leads_out:
            if self.model_params['num_leads'] == 8:
              leads_out = u.ECG_8_LEADS
            elif self.model_params['num_leads'] == 12:
              leads_out = u.ECG_12_LEADS
        self.leads_in = leads_in # list of lead names for ecg dataset
        self.leads_out = leads_out # list of lead names required for the model

    """
    Function to generate feat. 
    """
    def get_feats(self):
        init = time.time()
        hf_feat = h5py.File(self.feat_path, 'a')
        for loop in np.arange(self.num_slices):
          ecg = np.array(self.data[(self.sl*loop):((self.sl*loop)+self.sl)])
          if loop ==0: print(f'array of shape ={ecg.shape} loaded from {self.data_dir} in {time.time()-init:.4f} sec')
          ecg = u.ecg_process(ecg, self.leads_in, self.leads_out, 
                            ecg_sampling= self.data_info['sampling'][self.dataset],
                            filt=self.data_params['filt'], sampling_rate=self.model_params['sampling_rate'], 
                            num_samples=self.model_params['num_samples'], pad=self.data_params['pad'])

          if loop == 0: print(f'processed to shape ={ecg.shape} in {time.time()-init:.4f} sec')
          if self.model_params['power']: ecg = np.sign(ecg) * (np.abs(ecg)) ** self.power
          feats = self.ecg_model.predict(ecg) 
          if loop == 0: print(f'predicted for {ecg.shape} in {time.time()-init:.4f} sec')
          if loop==0:
            hf_feat.create_dataset('feats', data=feats, chunks=True, maxshape=(None , 320)) 
          else:
            hf_feat['feats'].resize((hf_feat['feats'].shape[0] + feats.shape[0]), axis=0)
            hf_feat['feats'][-feats.shape[0]:] = feats
          feat_shape = hf_feat['feats'].shape
          print(f'Iteration {loop} / {self.num_slices} : {feat_shape} : {time.time()-init:.4f} sec')
        print('Completed')
        hf_feat.close()
        print(f'The ECGs are successfully saved to {self.feat_path} in {time.time()-init:.4f} sec')

Create CAPE object.

In [6]:
cape = CAPE(dataset='ptb', name = 'mimic', model = 'CAPE_mimic_2',)

2025-11-04 15:19:14.646574: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-04 15:19:14.646639: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (cx3-15-29.cx3.hpc.ic.ac.uk): /proc/driver/nvidia/version does not exist
2025-11-04 15:19:14.647628: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 15:19:14.869069: W tensorflow/core/util/tensor_slice_reader.cc:96] Could not open models/CAPE_mimic_2/best_model: FAILED_PRECONDITION: models/CAPE_mimic_2/best_model; Is a directory: perhaps your file is in a different file format and you need to use a different resto

ecg data of shape = (21799, 5000, 12) found at data/ECGdata/ptb
The data of size 21799 will be processed in 22 slices of 1000


Generate features.

In [7]:
cape.get_feats()

array of shape =(1000, 5000, 12) loaded from data/ECGdata/ptb in 2.2951 sec
processed to shape =(1000, 2800, 8) in 21.7414 sec
predicted for (1000, 2800, 8) in 98.7407 sec
Iteration 0 / 22 : (1000, 256) : 98.7443 sec
Iteration 1 / 22 : (2000, 256) : 199.5065 sec
Iteration 2 / 22 : (3000, 256) : 305.6092 sec
Iteration 3 / 22 : (4000, 256) : 402.7335 sec
Iteration 4 / 22 : (5000, 256) : 498.6291 sec
Iteration 5 / 22 : (6000, 256) : 590.3140 sec
Iteration 6 / 22 : (7000, 256) : 682.0356 sec
Iteration 7 / 22 : (8000, 256) : 776.3305 sec
Iteration 8 / 22 : (9000, 256) : 868.5265 sec
Iteration 9 / 22 : (10000, 256) : 968.3074 sec
Iteration 10 / 22 : (11000, 256) : 1063.1395 sec
Iteration 11 / 22 : (12000, 256) : 1156.6356 sec
Iteration 12 / 22 : (13000, 256) : 1249.2351 sec
Iteration 13 / 22 : (14000, 256) : 1344.9271 sec
Iteration 14 / 22 : (15000, 256) : 1433.7054 sec
Iteration 15 / 22 : (16000, 256) : 1527.1298 sec
Iteration 16 / 22 : (17000, 256) : 1623.0412 sec
Iteration 17 / 22 : (1800

Use for saving the backbone as h5

In [ ]:
#model_path = os.path.join('models', f'Saved_{cape.model}', 'best_model.h5')
#cape.ecg_model.save(model_path )